# img2pcd 스파이크 — 이미지 1장 → TRELLIS.2 → PCD

**목적**: 자동차 범퍼 커버 / 본네트 이미지를 3D 로 만들어 `.pcd` 로 뽑는 게 되는지 입증한다.
spec(`docs/bumper-synth-spec.md`)의 FR/NFR 은 따르지 않는다 — DB·run·라벨 없음.

**순서**: 진단(1) → 후반부 검증(2) → 설치(3) → 입력(4) → 로드(5) → 생성(6) → 결과(7~8)

**입력**: `inputs/bumper.jpg`, `inputs/hood.jpg` (배경 단순 + 부품 하나만 크게)

**출력**: `out/<부품>/{mesh.glb, mesh_mm.ply, <부품>.pcd, preview.png, manifest.json}`

**필요 환경**: Linux · NVIDIA 24GB+ · CUDA toolkit(nvcc) · conda · 디스크 100GB+
(PyTorch 는 `setup_vast.sh` 가 설치한다)

## 1. 설치 전 환경 진단

**FAIL 이 있으면 여기서 멈춘다.** 이대로 설치를 돌리면 도중에 깨진다.
표준 라이브러리만 쓰므로 아무것도 안 깔린 base 이미지에서도 돈다.

가장 흔한 실패는 **`nvcc` 없음** — CuMesh·o-voxel·nvdiffrast 가 소스 빌드라
vast.ai 에서 `*-runtime` 이미지를 고르면 여기서 걸린다. `*-devel` 이나 PyTorch 템플릿을 쓸 것.

> **PyTorch 는 미리 깔 필요 없다.** `setup.sh --new-env` 가 conda env `trellis2` 를 만들고
> 거기에 `torch==2.6.0` (cu124) 을 직접 설치한다. 준비할 건 CUDA **toolkit** 과 conda 다.

In [ ]:
!python check_env.py

## 2. GPU 없이 후반부만 먼저 검증 (선택)

합성 메시로 스케일→샘플링→PCD→미리보기를 돌린다. TRELLIS.2 설치 전에도 된다.

In [ ]:
!python smoke_test.py

## 3. TRELLIS.2 설치

20~40분 걸린다(모델 ~15GB + 의존성 ~30GB). **터미널에서 돌리는 걸 권장** —
노트북 셀은 중간에 끊기면 되살리기 어렵고, `conda activate` 가 셀 사이에 유지되지 않는다.

```bash
bash setup_vast.sh
```

끝나면 **커널을 `Python (trellis2)` 로 바꾸고** 아래 셀로 설치를 검증한다.
(Jupyter 상단 커널 선택 → Python (trellis2). 안 보이면 브라우저 새로고침)

In [ ]:
# 설치 후 검증 — 커널이 Python (trellis2) 인지부터 확인
import sys
print('python  :', sys.executable, '   <- .../envs/trellis2/bin/python 이어야 정상')

import torch
print('torch   :', torch.__version__, '| cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print('gpu     :', p.name, f'{p.total_memory / 2**30:.1f} GiB')

print('\n임포트 검증:')
missing = []
for m in ('trellis2', 'o_voxel', 'trimesh', 'open3d', 'yaml', 'matplotlib'):
    try:
        __import__(m)
        print(f'  OK    {m}')
    except Exception as e:
        print(f'  FAIL  {m}: {e}')
        missing.append(m)

print('\n=> 설치 완료' if not missing else f'\n=> 실패: {missing} — setup_vast.sh 로그를 확인하세요')

## 4. 입력 이미지 확인

`inputs/` 에 이미지를 올린 뒤 실행. Jupyter 파일 브라우저로 드래그 업로드하면 된다.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image
import img2pcd

parts, settings = img2pcd.load_config(Path('parts.yaml'))
fig, axes = plt.subplots(1, len(parts), figsize=(6 * len(parts), 5))
axes = [axes] if len(parts) == 1 else list(axes)
for ax, part in zip(axes, parts):
    p = Path(part.image)
    ax.axis('off')
    if p.exists():
        ax.imshow(Image.open(p))
        ax.set_title(f'{part.name}  target={part.target_mm:.0f}mm')
    else:
        ax.set_title(f'{part.name} — 이미지 없음: {p}')
plt.tight_layout(); plt.show()
print(settings)

## 5. 파이프라인 로드

첫 실행은 가중치 ~15GB 다운로드라 오래 걸린다.
`run()` 시그니처를 찍어두면 이 버전이 실제로 받는 인자(seed·resolution 등)를 확인할 수 있다.

In [ ]:
# settings.run_kwargs = {'resolution': 512}   # 5번 셀에서 확인한 실제 인자 이름으로

out_root = Path('out').resolve()
results = []
for part in parts:
    results.append(img2pcd.process_part(part, settings, out_root, pipeline, base_dir=Path.cwd()))

## 6. 부품별 생성

위 셀에서 본 인자 이름에 맞춰 `settings.run_kwargs` 를 채우면 해상도 등을 바꿀 수 있다.
받지 않는 인자는 자동으로 걸러진다.

> ⚠️ 24GB 급에서 고해상도(1024³/1536³)는 메시 후처리에서 OOM 이 보고돼 있다
> ([이슈 #188](https://github.com/microsoft/TRELLIS.2/issues/188)). 기본값으로 먼저 돌리고,
> 터지면 해상도를 낮추거나 부품을 하나씩 돌린다.

In [ ]:
# settings.run_kwargs = {'resolution': 512}   # 4번 셀에서 확인한 이름으로

out_root = Path('out').resolve()
results = []
for part in parts:
    results.append(img2pcd.process_part(part, settings, out_root, pipeline, base_dir=Path.cwd()))

## 7. 결과 — 3면도 미리보기

In [ ]:
from IPython.display import display, Image as IPImage
for r in results:
    print(f"=== {r['part']} ===")
    display(IPImage(filename=str(out_root / r['part'] / 'preview.png')))

## 8. 입증 요약 — PCD 를 다시 읽어 검증

In [ ]:
import open3d as o3d

print(f"{'부품':14s} {'점 개수':>10s} {'색':>4s} {'bbox_mm (x,y,z)':>28s} {'VRAM피크':>9s} {'생성s':>7s}")
print('-' * 80)
for r in results:
    v = r['verify']
    pcd = o3d.io.read_point_cloud(v['path'])          # 파일에서 실제로 다시 읽는다
    bbox = ', '.join(f'{b:.0f}' for b in v['bbox_mm'])
    print(f"{r['part']:14s} {len(pcd.points):>10,} {str(pcd.has_colors()):>4s} "
          f"{bbox:>28s} {r.get('peak_vram_gib', '-'):>9} {r.get('t_generate_s', '-'):>7}")

print('\n산출 파일:')
for f in sorted(out_root.rglob('*')):
    if f.is_file():
        print(f'  {f.relative_to(out_root)}  ({f.stat().st_size / 2**20:.2f} MB)')

## 9. 다음 단계

PCD 가 쓸 만하면 프로젝트로 가져가 등록한다(스파이크는 여기까지):

```bash
bumper asset add out/bumper_cover/bumper_cover.pcd --name trellis_bumper_01 --part bumper_cover --unit mm
bumper asset add out/hood/hood.pcd --name trellis_hood_01 --part hood --unit mm
```

판단해야 할 것:
1. **형상이 실제 부품 같은가** — 생성 모델은 없는 형상을 지어낸다
2. **표면이 매끄러운가** — 결함 깊이가 0.1~1mm 급인데 배경 표면이 그만큼 울퉁불퉁하면 학습 신호가 오염된다.
   심하면 `--smooth 5~10` 으로 Taubin 스무딩을 건다
3. **스케일이 맞는가** — `target_mm` 을 실측값으로 넣었는지. 이건 사람이 넣는 값이라 자동 검증 불가